# Классификация страхователей транспортных средств

Соревнование Kaggle: [findata-policyholders](https://www.kaggle.com/competitions/findata-policyholders).
Задача — бинарная классификация клиентов страховой компании по табличным данным. Метрика — ROC-AUC.

**План:**
1. Загрузка и первичный анализ данных
2. Разбиение на train / valid / test **до** любой обработки (без утечки данных)
3. Предобработка пропусков и категориальных признаков
4. Обучение CatBoost с учётом дисбаланса классов и ранней остановкой
5. Подбор порога классификации на валидации, итоговая оценка на отложенном test
6. Анализ важности признаков и формирование сабмита

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score, accuracy_score, precision_score, recall_score,
    f1_score, classification_report, confusion_matrix
)
from catboost import CatBoostClassifier, Pool

RANDOM_STATE = 123
DATA_DIR = Path('data')  # положите сюда файлы соревнования

## 1. Загрузка и первичный анализ

In [ ]:
data = pd.read_csv(DATA_DIR / 'insclass_train.csv')
print('Размер train:', data.shape)
data.head()

In [ ]:
# Баланс классов
print(data['target'].value_counts(normalize=True).round(3))
data['target'].value_counts().plot(kind='bar', title='Распределение целевой переменной');

In [ ]:
# Доля пропусков по признакам
missing = data.isna().mean().sort_values(ascending=False)
missing[missing > 0].round(3)

## 2. Разбиение данных

Делим данные на три части **до** заполнения пропусков, чтобы статистики не «подсматривали» в валидацию и тест:
- **train (70%)** — обучение модели;
- **valid (15%)** — ранняя остановка и подбор порога;
- **test (15%)** — финальная честная оценка качества.

In [ ]:
X = data.drop(columns=['target'])
y = data['target']

X_train, X_tmp, y_train, y_tmp = train_test_split(
    X, y, test_size=0.30, random_state=RANDOM_STATE, stratify=y)
X_valid, X_test, y_valid, y_test = train_test_split(
    X_tmp, y_tmp, test_size=0.50, random_state=RANDOM_STATE, stratify=y_tmp)

print(f'train: {X_train.shape}, valid: {X_valid.shape}, test: {X_test.shape}')

## 3. Предобработка

- Категориальные признаки: пропуски заменяем отдельной категорией `missing` и передаём в CatBoost как категориальные.
- Бинарные признаки: пропуск трактуем как 0.
- Числовые признаки: оставляем `NaN` — CatBoost обрабатывает пропуски сам, поэтому статистики по всему датасету не нужны.

In [ ]:
CAT_FEATURES = ['variable_1', 'variable_5', 'variable_20',
                'variable_21', 'variable_22', 'variable_28']
BINARY_COLS = ['variable_2', 'variable_3', 'variable_10', 'variable_11',
               'variable_23', 'variable_25', 'variable_26']

def preprocess(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for col in CAT_FEATURES:
        if col in df.columns:
            df[col] = df[col].fillna('missing').astype(str)
    for col in BINARY_COLS:
        if col in df.columns:
            df[col] = df[col].fillna(0).astype(int)
    return df

X_train, X_valid, X_test = map(preprocess, (X_train, X_valid, X_test))

## 4. Обучение CatBoost

In [ ]:
params = dict(
    iterations=2000,
    learning_rate=0.05,
    depth=6,
    auto_class_weights='Balanced',  # учитываем дисбаланс классов
    loss_function='Logloss',
    eval_metric='AUC',
    random_seed=RANDOM_STATE,
    verbose=200,
)

model = CatBoostClassifier(**params)
model.fit(
    Pool(X_train, y_train, cat_features=CAT_FEATURES),
    eval_set=Pool(X_valid, y_valid, cat_features=CAT_FEATURES),
    early_stopping_rounds=100,
    use_best_model=True,
)
print('Лучшая итерация:', model.get_best_iteration())

## 5. Подбор порога и оценка качества

ROC-AUC не зависит от порога. Порог нужен для бинарных меток (F1, precision, recall), его подбираем **только на valid**.

In [ ]:
valid_proba = model.predict_proba(X_valid)[:, 1]
thresholds = np.arange(0.05, 0.95, 0.01)
f1_scores = [f1_score(y_valid, (valid_proba >= t).astype(int)) for t in thresholds]
best_threshold = thresholds[int(np.argmax(f1_scores))]
print(f'Лучший порог по F1 на valid: {best_threshold:.2f}')

plt.plot(thresholds, f1_scores)
plt.axvline(best_threshold, ls='--', c='gray')
plt.xlabel('Порог'); plt.ylabel('F1'); plt.title('Подбор порога на valid');

In [ ]:
test_proba = model.predict_proba(X_test)[:, 1]
test_pred = (test_proba >= best_threshold).astype(int)

print('Метрики на отложенном test')
print(f'ROC-AUC:   {roc_auc_score(y_test, test_proba):.4f}')
print(f'Accuracy:  {accuracy_score(y_test, test_pred):.4f}')
print(f'Precision: {precision_score(y_test, test_pred):.4f}')
print(f'Recall:    {recall_score(y_test, test_pred):.4f}')
print(f'F1-score:  {f1_score(y_test, test_pred):.4f}')
print()
print(classification_report(y_test, test_pred))

In [ ]:
sns.heatmap(confusion_matrix(y_test, test_pred), annot=True, fmt='d', cmap='Blues')
plt.title('Матрица ошибок (test)'); plt.xlabel('Предсказание'); plt.ylabel('Факт');

## 6. Важность признаков

In [ ]:
imp = (pd.DataFrame({'feature': X_train.columns,
                     'importance': model.get_feature_importance()})
       .sort_values('importance', ascending=False).head(15))
plt.figure(figsize=(9, 6))
sns.barplot(data=imp, x='importance', y='feature')
plt.title('Топ-15 признаков по важности');

## 7. Финальная модель и сабмит

Переобучаем модель на всех размеченных данных с найденным числом итераций.
Так как метрика соревнования — ROC-AUC, в сабмит отправляем **вероятности**, а не метки 0/1.

In [ ]:
X_full = preprocess(X)
final_params = {**params, 'iterations': model.get_best_iteration() + 1, 'verbose': 0}
final_model = CatBoostClassifier(**final_params)
final_model.fit(Pool(X_full, y, cat_features=CAT_FEATURES))

test_data = preprocess(pd.read_csv(DATA_DIR / 'insclass_test.csv'))
proba = final_model.predict_proba(test_data[X_full.columns])[:, 1]

# Формат сабмита — по образцу sample_submission.csv из соревнования
sample_path = DATA_DIR / 'sample_submission.csv'
if sample_path.exists():
    submission = pd.read_csv(sample_path)
    submission[submission.columns[-1]] = proba
else:
    submission = pd.DataFrame({'target': proba})
submission.to_csv('submission.csv', index=False)
submission.head()

## Итоги

- Публичный лидерборд: **ROC-AUC 0.6727**, 134-е место из 666 участников (топ-21%) с первой попытки; у лидера — 0.6849.
- В этой версии устранена утечка данных (обработка и подбор порога только на train/valid), добавлен честный test и сабмит вероятностей.

**Что можно улучшить:** генерация признаков, подбор гиперпараметров (Optuna), кросс-валидация, ансамбль с LightGBM.